In [1]:
## Cell 1 — Setup & Load Embedding Model + DocumentStore (Lazy Loading)

import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from FlagEmbedding import BGEM3FlagModel
from rank_bm25 import BM25Okapi
from collections import defaultdict

# ── Config ─────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent
VECTOR_DB_DIR = PROJECT_ROOT / "data" / "src" / "vector_db"
DOCLING_OUTPUT = PROJECT_ROOT / "data" / "src" / "docling_output"
BGE_M3_MODEL = "BAAI/bge-m3"

# ── Docling Output Verzeichnis (explizit, nicht auto-latest) ──
DOCLING_RUN = "20260428_142906"
PAGES_DIR = DOCLING_OUTPUT / f"{DOCLING_RUN}_merged"
assert PAGES_DIR.exists(), f"Parent pages dir not found: {PAGES_DIR}"

# ── Ändere TOP_PAGES hier (3 oder 10) ────────────────────
TOP_PAGES = 10
# ────────────────────────────────────────────────────────

TOP_K = 25          # chunks from dense retrieval (for eval display)
RRF_K = 60          # RRF fusion parameter
NUM_FEW_SHOT = 3    # number of dynamic few-shot examples to inject

# ── Load embedding model ────────────────────────────────────────
embed_model = BGEM3FlagModel(BGE_M3_MODEL, use_fp16=True)

# ── Helper functions ────────────────────────────────────────────

def sparse_sim(q_sparse: dict, d_sparse: dict) -> float:
    """Dot product of shared lexical token weights."""
    score = 0.0
    for token, qw in q_sparse.items():
        if token in d_sparse:
            score += float(qw) * float(d_sparse[token])
    return score


def colbert_maxsim(q_colbert: np.ndarray, d_colbert: np.ndarray) -> float:
    """ColBERT MaxSim: sum of max cosine similarity per query token."""
    if q_colbert is None or d_colbert is None or len(q_colbert) == 0 or len(d_colbert) == 0:
        return 0.0
    q_norm = q_colbert / (np.linalg.norm(q_colbert, axis=1, keepdims=True) + 1e-8)
    d_norm = d_colbert / (np.linalg.norm(d_colbert, axis=1, keepdims=True) + 1e-8)
    sim = q_norm @ d_norm.T          # (q_tokens x d_tokens)
    return float(sim.max(axis=1).sum())


def rrf_fuse(rankings: list[list], k: int = RRF_K) -> tuple[list, dict]:
    """Reciprocal Rank Fusion of multiple page rankings. Returns (sorted_keys, scores_dict)."""
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, item in enumerate(ranking):
            scores[item] += 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True), dict(scores)


# ── DocumentStore: Lazy Loading mit LRU-Cache ───────────────

class DocumentStore:
    """Loads vector DB + parent pages on demand, evicts old docs via LRU."""

    def __init__(self, vector_db_dir: Path, pages_dir: Path, max_cached: int = 5):
        self._vector_db_dir = vector_db_dir
        self._pages_dir = pages_dir
        self._max_cached = max_cached
        self._cache_order = []  # LRU tracking

        # Lazy-loaded per doc
        self._metadata = {}
        self._dense_vecs = {}
        self._colbert_vecs = {}
        self._sparse_vecs = {}
        self._bm25 = {}
        self._parent_pages = {}

        # Discover available docs (just file listing, no loading)
        self._available_docs = {
            fp.stem.replace("_meta", "")
            for fp in self._vector_db_dir.glob("*_meta.json")
        }

    @property
    def available_docs(self) -> set:
        return self._available_docs

    def _evict_if_needed(self):
        """Remove oldest doc from cache when max reached."""
        while len(self._cache_order) > self._max_cached:
            old = self._cache_order.pop(0)
            self._metadata.pop(old, None)
            self._dense_vecs.pop(old, None)
            self._colbert_vecs.pop(old, None)
            self._sparse_vecs.pop(old, None)
            self._bm25.pop(old, None)
            self._parent_pages.pop(old, None)

    def _touch(self, doc_name: str):
        """Mark doc as recently used."""
        if doc_name in self._cache_order:
            self._cache_order.remove(doc_name)
        self._cache_order.append(doc_name)
        self._evict_if_needed()

    def _load_vectors(self, doc_name: str):
        """Load vector DB files for one document."""
        if doc_name in self._metadata:
            return
        vdb = self._vector_db_dir
        with open(vdb / f"{doc_name}_meta.json", "r", encoding="utf-8") as f:
            self._metadata[doc_name] = json.load(f)
        self._dense_vecs[doc_name] = np.load(str(vdb / f"{doc_name}_dense.npy"))
        colbert_data = np.load(str(vdb / f"{doc_name}_colbert.npz"), allow_pickle=True)
        self._colbert_vecs[doc_name] = list(colbert_data["vecs"])
        with open(vdb / f"{doc_name}_sparse.pkl", "rb") as f:
            self._sparse_vecs[doc_name] = pickle.load(f)
        # BM25 index
        corpus = [m["text"].lower().split() for m in self._metadata[doc_name]]
        self._bm25[doc_name] = BM25Okapi(corpus)

    def _load_pages(self, doc_name: str):
        """Load parent pages for one document."""
        if doc_name in self._parent_pages:
            return
        fp = self._pages_dir / f"{doc_name}.json"
        if not fp.exists():
            self._parent_pages[doc_name] = {}
            return
        with open(fp, "r", encoding="utf-8") as f:
            doc = json.load(f)
        pages_list = doc.get("content", {}).get("pages", []) or []
        self._parent_pages[doc_name] = {p["page"]: p["text"] for p in pages_list}

    def get(self, doc_name: str) -> dict:
        """Load (if needed) and return all data for a document."""
        if doc_name not in self._available_docs:
            raise ValueError(f"No vector DB data for document: {doc_name}")
        self._load_vectors(doc_name)
        self._load_pages(doc_name)
        self._touch(doc_name)
        return {
            "metadata": self._metadata[doc_name],
            "dense_vecs": self._dense_vecs[doc_name],
            "colbert_vecs": self._colbert_vecs[doc_name],
            "sparse_vecs": self._sparse_vecs[doc_name],
            "bm25": self._bm25[doc_name],
            "parent_pages": self._parent_pages[doc_name],
        }

    def has_parent_pages(self, doc_name: str) -> bool:
        """Check if parent pages file exists (without loading)."""
        return (self._pages_dir / f"{doc_name}.json").exists()

    @property
    def cached_docs(self) -> list:
        return list(self._cache_order)


# ── Instantiate DocumentStore ───────────────────────────────
doc_store = DocumentStore(VECTOR_DB_DIR, PAGES_DIR, max_cached=5)

print(f"DocumentStore ready: {len(doc_store.available_docs)} documents available (lazy loading, max 5 cached)")
print(f"  Vector DB: {VECTOR_DB_DIR}")
print(f"  Pages:     {PAGES_DIR}")

C:\Users\phili\PycharmProjects\Retrieval_Program_of_Thought\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 30 files: 100%|██████████| 30/30 [00:00<?, ?it/s]


DocumentStore ready: 100 documents available (lazy loading, max 5 cached)
  Vector DB: C:\Users\phili\PycharmProjects\Retrieval_Program_of_Thought\data\src\vector_db
  Pages:     C:\Users\phili\PycharmProjects\Retrieval_Program_of_Thought\data\src\docling_output\20260428_142906_merged


In [2]:
## Cell 2 — Load QA Benchmark
QA_DIR = PROJECT_ROOT / "data" / "qa"

df = pd.read_csv(QA_DIR / "fin_docs_train_questions.csv", sep="|")
df["source"] = "fin"

# Drop questions where both answers are NaN
before = len(df)
df = df.dropna(subset=["answer_1", "answer_2"], how="all").reset_index(drop=True)
print(f"Dropped {before - len(df)} questions with no ground-truth answer")

# Extract ground-truth page from q_uid  (e.g. 'ADI/2009/page_49.pdf-1' → 'ADI/2009/page_49.pdf')
df["gt_pdf"] = df["q_uid"].str.rsplit("-", n=1).str[0]

print(f"Loaded {len(df)} questions (fin only)")
print(f"Documents: {sorted(df['doc_name'].unique())}")
print()
df[["doc_name", "q_uid", "gt_pdf", "question"]].head(10)

Dropped 0 questions with no ground-truth answer
Loaded 895 questions (fin only)
Documents: ['AAL_2017', 'AAPL_2002', 'AAPL_2004', 'AAPL_2007', 'AAPL_2011', 'AAPL_2013', 'ABMD_2015', 'ADBE_2008', 'ADI_2009', 'AES_2002', 'ALLE_2016', 'ALXN_2007', 'AMT_2003', 'AMT_2004', 'AMT_2007', 'AMT_2014', 'AMT_2015', 'ANET_2017', 'AON_2014', 'AON_2018', 'APTV_2014', 'APTV_2016', 'APTV_2018', 'BDX_2018', 'BKR_2017', 'BLK_2012', 'BLK_2014', 'BLK_2017', 'BLL_2006', 'BLL_2007', 'BLL_2010', 'CAG_2007', 'CAG_2008', 'CB_2010', 'CDNS_2006', 'CDW_2015', 'CE_2017', 'C_2010', 'C_2017', 'C_2018', 'DISH_2014', 'DRE_2016', 'DVN_2014', 'ETFC_2007', 'ETR_2017', 'FIS_2007', 'GPN_2008', 'GPN_2017', 'HIG_2013', 'HII_2015', 'HST_2018', 'HWM_2015', 'INTC_2013', 'INTC_2015', 'IPG_2012', 'IPG_2014', 'IPG_2016', 'IP_2006', 'IQV_2016', 'JKHY_2016', 'JPM_2010', 'JPM_2015', 'JPM_2016', 'K_2012', 'K_2013', 'LMT_2013', 'LMT_2016', 'MMM_2013', 'MO_2014', 'MRO_2007', 'MRO_2013', 'MRO_2014', 'MSI_2005', 'MSI_2009', 'PKG_2001', 'PK

,doc_name,q_uid,gt_pdf,question
0,AAL_2017,AAL/2017/page_33.pdf-1,AAL/2017/page_33.pdf,did american have access to more planes than a...
1,AAL_2017,AAL/2017/page_10.pdf-1,AAL/2017/page_10.pdf,based on the information provided what was the...
2,AAL_2017,AAL/2017/page_10.pdf-4,AAL/2017/page_10.pdf,what are the total operating expenses based on...
3,AAL_2017,AAL/2017/page_10.pdf-2,AAL/2017/page_10.pdf,as of 2017 what was the total annual fuel expe...
4,AAL_2017,AAL/2017/page_10.pdf-3,AAL/2017/page_10.pdf,what is the percentage change in the average p...
5,AAL_2017,AAL/2017/page_33.pdf-2,AAL/2017/page_33.pdf,what percent of american's total planes carrie...
6,AAPL_2002,AAPL/2002/page_26.pdf-4,AAPL/2002/page_26.pdf,what was the change in millions of total other...
7,AAPL_2002,AAPL/2002/page_63.pdf-3,AAPL/2002/page_63.pdf,what was the increase in total minimum lease p...
8,AAPL_2002,AAPL/2002/page_26.pdf-3,AAPL/2002/page_26.pdf,what was the greatest amount of total other in...
9,AAPL_2002,AAPL/2002/page_63.pdf-4,AAPL/2002/page_63.pdf,what percentage of total minimum lease payment...


In [3]:
# ── Ab welcher Frage starten? (1-based, 1 = alle) ──
SKIP_TO = 1
df = df.iloc[SKIP_TO - 1:].reset_index(drop=True)
print(f"Starting from question {SKIP_TO}, {len(df)} questions remaining")

Starting from question 1, 895 questions remaining


In [4]:
## Cell 3 — Few-Shot Example Pool + Embeddings (PoT)

FEW_SHOT_POOL = [
    # ── Percentage change ──
    {
        "question": "what is the percent change in total recurring capital expenditures from 2066 to 2067?",
        "answer": "6.54",
        "code": "old = 1000\nnew = 1065.4\nprint(round((new - old) / abs(old) * 100, 2))",
    },
    {
        "question": "what is the percentage change in the balance of non-controlling interests from 2060 to 2061?",
        "answer": "6.23",
        "code": "old = 10000\nnew = 10623\nprint(round((new - old) / abs(old) * 100, 2))",
    },
    # ── Percentage of / portion ──
    {
        "question": "what is the long term debt as a percentage of total contractual obligations in 2067?",
        "answer": "66.41",
        "code": "long_term_debt = 6641\ntotal_obligations = 10000\nprint(round(long_term_debt / total_obligations * 100, 2))",
    },
    {
        "question": "what are the total real estate investments as a percentage of the total assets acquired?",
        "answer": "67.18",
        "code": "real_estate = 6718\ntotal_assets = 10000\nprint(round(real_estate / total_assets * 100, 2))",
    },
    # ── Lookup ──
    {
        "question": "what is the net income per common share in 2067?",
        "answer": "6.56",
        "code": "net_income_per_share = 6.56\nprint(net_income_per_share)",
    },
    # ── Sum / total ──
    {
        "question": "what was the total fees earned in 2066 for management, leasing and construction and development?",
        "answer": "64.9",
        "code": "management = 20.5\nleasing = 30.2\nconstruction = 14.2\nprint(round(management + leasing + construction, 1))",
    },
    # ── Cumulative return ──
    {
        "question": "what was the percentage cumulative total shareholder return on disca for the five year period ended december 21, 2016?",
        "answer": "568.56",
        "code": "beginning = 100\nending = 668.56\nprint(round((ending / beginning - 1) * 100, 2))",
    },
    # ── Difference ──
    {
        "question": "what was the difference in millions of international subscribers between discovery channel and animal planet?",
        "answer": "76",
        "code": "discovery_channel = 300\nanimal_planet = 224\nprint(discovery_channel - animal_planet)",
    },
    # ── Average ──
    {
        "question": "what was the average dividend per share from 2066 to 2068 in dollars per share?",
        "answer": "6.91",
        "code": "div_2066 = 6.5\ndiv_2067 = 7.0\ndiv_2068 = 7.23\nprint(round((div_2066 + div_2067 + div_2068) / 3, 2))",
    },
    # ── Ratio ──
    {
        "question": "in 2062 what was the ratio of the eligibility limits for farmer and cooperative to individual participants in the family?",
        "answer": "6.00",
        "code": "farmer_coop = 300000\nindividual = 50000\nprint(round(farmer_coop / individual, 2))",
    },
    # ── Unrealized gain (multi-step) ──
    {
        "question": "what is the unrealized gain pre-tax for bolsa mexicana de valores?",
        "answer": "16",
        "code": "fair_value = 66\ncost = 50\nprint(fair_value - cost)",
    },
    # ── Implied value (multi-step) ──
    {
        "question": "as of january 21, 2014, what was the implied total value of eurosport international based on the price paid for the interest acquired?",
        "answer": "1606.45",
        "code": "price_paid = 481.935\ninterest_pct = 30\nprint(round(price_paid / (interest_pct / 100), 2))",
    },
]

# ── Pre-compute embeddings for few-shot questions ──────────
pool_questions = [ex["question"] for ex in FEW_SHOT_POOL]
pool_enc = embed_model.encode(pool_questions, return_dense=True, return_sparse=False, return_colbert_vecs=False)
pool_embeddings = np.array(pool_enc["dense_vecs"], dtype=np.float32)
pool_embeddings = pool_embeddings / (np.linalg.norm(pool_embeddings, axis=1, keepdims=True) + 1e-8)

print(f"Few-shot pool: {len(FEW_SHOT_POOL)} examples, embeddings shape={pool_embeddings.shape}")


def select_few_shot(query_dense: np.ndarray, n: int = NUM_FEW_SHOT) -> list[dict]:
    """Select n most similar few-shot examples via cosine similarity."""
    sims = (query_dense @ pool_embeddings.T)[0]
    top_idx = np.argsort(sims)[::-1][:n]
    return [FEW_SHOT_POOL[i] for i in top_idx]


def format_few_shot(examples: list[dict]) -> str:
    """Format selected few-shot examples with code for PoT prompt."""
    parts = []
    for i, ex in enumerate(examples):
        parts.append(
            f"### Example {i+1}\n"
            f"Question: {ex['question']}\n"
            f"```python\n{ex['code']}\n```\n"
            f"Answer: {ex['answer']}"
        )
    return "\n\n".join(parts)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Few-shot pool: 12 examples, embeddings shape=(12, 1024)


In [5]:
## Cell 4 — `ask()`: 2-Step PoT Pipeline (Fact Extraction → Code Generation → Execution)

import requests
import jsonlines
import re
import io
import sys
import threading
from datetime import datetime

# ── LLM Config ───────────────────────────────────────────
LM_STUDIO_URL = "http://localhost:1234/v1/chat/completions"
LLM_MODEL = "google/gemma-4-26b-a4b"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def strip_think(text: str) -> str:
    """Remove <think>...</think> blocks from LLM output.

    If stripping would kill ALL content (i.e. the model emitted only a
    think block with no trailing answer, or an unclosed <think>), fall
    back to the raw output instead of returning empty. Call 2 can
    extract numbers from reasoning text — but cannot do anything with
    an empty string.
    """
    original = text
    text = re.sub(r"<think>[\s\S]*?</think>\s*", "", text).strip()
    text = re.sub(r"<think>[\s\S]*$", "", text).strip()
    if not text and original.strip():
        return original.strip()
    return text


def normalize_answer(val: str) -> str:
    """Normalize an answer string for comparison."""
    s = str(val).strip().lower()
    s = s.replace(",", "").replace("%", "").replace("$", "")
    s = s.replace(" ", "")
    if s.endswith(".0"):
        s = s[:-2]
    return s


def try_float(val: str):
    """Try to parse a string as float, return None on failure."""
    try:
        return float(normalize_answer(val))
    except (ValueError, TypeError):
        return None


def match_answer(
    llm_answer: str,
    gt_answers: list,
    rel_tol: float = 0.01,
) -> dict:
    """Compare LLM answer against ground truth. Returns exact_match, lenient_match."""
    pred = normalize_answer(llm_answer)

    exact = False
    for gt in gt_answers:
        if pred == normalize_answer(gt):
            exact = True
            break

    lenient = exact
    if not lenient:
        pred_f = try_float(llm_answer)
        if pred_f is not None:
            for gt in gt_answers:
                gt_f = try_float(gt)
                if gt_f is not None:
                    if gt_f == 0:
                        if pred_f == 0:
                            lenient = True
                            break
                    elif abs(pred_f - gt_f) / abs(gt_f) <= rel_tol:
                        lenient = True
                        break

    return {"exact_match": exact, "lenient_match": lenient}


# ── Code Extraction & Execution ───────────────────────────

def extract_code_block(text: str) -> str | None:
    """Extract Python code from a ```python ... ``` block in LLM response."""
    text = strip_think(text)
    match = re.search(r"```python\s*\n(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    match = re.search(r"```\s*\n(.*?)```", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None


def execute_code(code: str, timeout: int = 10) -> dict:
    """Execute Python code and capture stdout. Returns {success, output, error}."""
    result = {"success": False, "output": "", "error": None}
    output_buf = io.StringIO()

    def run():
        old_stdout = sys.stdout
        try:
            sys.stdout = output_buf
            exec(code, {"__builtins__": __builtins__})
            result["success"] = True
        except Exception as e:
            result["error"] = f"{type(e).__name__}: {e}"
        finally:
            sys.stdout = old_stdout

    thread = threading.Thread(target=run)
    thread.start()
    thread.join(timeout=timeout)

    if thread.is_alive():
        result["error"] = f"Timeout: code execution exceeded {timeout}s"
        return result

    result["output"] = output_buf.getvalue().strip()
    return result


# ── Step 1: Fact Extraction Prompt ────────────────────────

FACT_EXTRACTION_PROMPT = """ROLE: You are a text scanner. You are NOT a reasoner.
Your job: read each page once and list numbers that match the question.

OUTPUT — exactly this format, nothing else:

FACTS:
- <description> : <number> <unit>
- <description> : <number> <unit>

FORBIDDEN in your output:
- "Wait", "Let me re-read", "Actually", "Hmm", "Let me check again"
- Discussion of table layout, OCR, column alignment
- Any "Self-Correction" section
- Any text before "FACTS:"

If a number is ambiguous (e.g. could belong to two columns), pick the
most likely interpretation and move on. Do not deliberate.

Hard limits:
- Max 15 facts total.
- Max 1 line per fact.
- No explanation after FACTS: block.

Example:
Question: In which year did Fictional Bank have the highest loan losses?
Context: "Loan loss provisions were $45M in 2018, $62M in 2019, and
$38M in 2020. Total loans outstanding reached $12.4B by year-end 2020."

FACTS:
- Loan loss provisions 2018 : 45 million dollars
- Loan loss provisions 2019 : 62 million dollars
- Loan loss provisions 2020 : 38 million dollars"""


def extract_facts(context_str: str, question: str, temperature: float = 0.1) -> dict:
    """Step 1: Extract numerical facts from context (thinking disabled)."""
    user_prompt = f"Context:\n{context_str}\n\nQuestion: {question}\n\nFACTS:"

    resp = requests.post(
        LM_STUDIO_URL,
        json={
            "model": LLM_MODEL,
            "messages": [
                {"role": "system", "content": FACT_EXTRACTION_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            "temperature": temperature,
            "think": False,
        },
        timeout=600,
    )
    resp.raise_for_status()
    resp_json = resp.json()
    msg = resp_json["choices"][0]["message"]
    raw_facts_text = (msg.get("content") or "").strip()
    fact_thinking = msg.get("reasoning_content") or None
    facts_text = strip_think(raw_facts_text)
    usage = resp_json.get("usage", {})

    return {
        "facts": facts_text,
        "raw_facts": raw_facts_text,
        "thinking": fact_thinking,
        "prompt_tokens": usage.get("prompt_tokens", 0),
        "completion_tokens": usage.get("completion_tokens", 0),
    }


# ── Step 2: Code Generation System Prompt (PoT) ───────────

SYSTEM_PROMPT_BASE = """You are a financial analyst answering questions about corporate 10-K annual reports.
Answer ONLY based on the extracted facts provided.

## Task
Write Python code that computes the answer and prints it with a single `print()` statement at the end.
Wrap your code in a ```python code block.

## Rules
1. Check fact descriptions carefully to identify the correct entity, line item, or segment. If the question names a specific entity, use that entity's data.
2. Assign the relevant numbers from the facts to descriptive variable names, then compute the answer.
3. End with exactly ONE `print()` call that outputs the final answer — a single number, no units, no currency symbols.
   - Use `round()` to control decimal places where appropriate.
   - Percentages: compute as percentage value (e.g. `print(19.9)`), NEVER as decimal (e.g. 0.199).
   - Ratios: round to 2 decimal places.
   - Yes/no questions: `print('yes')` or `print('no')`.
4. Only use Python builtins and basic arithmetic — no imports needed.
5. Unit scale: match the unit the question asks for.
   - "in millions" → value in millions (e.g. 471, not 471000000).
   - "in billions" → value in billions.
   - If the question specifies no unit, use the unit from the source data.
6. Sign convention:
   - "decrease", "decline", "loss", "reduction" → result should be negative.
   - "increase", "growth", "gain" → result should be positive.
   - "change" or "difference" with no direction word → compute (later_period − earlier_period).
   - "by how much did X change" → percentage change: ((new − old) / abs(old)) × 100.
7. Calculation patterns:
   - Percentage change: `((new - old) / abs(old)) * 100`
   - "what portion", "what percent of", "how much of X is Y" → `(part / whole) * 100`
   - "total" → sum the components. "average" → sum / count.
   - Cumulative stock return: `((ending / beginning) - 1) * 100`
   - Average uncompounded annual return: `cumulative_return / number_of_years`
   - Profit margins: use the line items as labeled.
8. Before printing INSUFFICIENT CONTEXT, re-read ALL extracted facts. The answer may require calculation from available numbers. Only print "INSUFFICIENT CONTEXT" if the required data is genuinely absent.
9. Denominator subset check: before adding two items in a denominator, check whether one is a subset of the other (standard accounting hierarchy).
   If Y ⊂ X, use X alone as the denominator — NOT X + Y.
   Common subset relationships:
   - Cash, Receivables, Inventory, Prepaids ⊂ Total Current Assets
   - Current Assets, Non-current Assets ⊂ Total Assets
   - Short-term Debt, Long-term Debt ⊂ Total Debt
   - Cost of Goods Sold, SG&A, R&D ⊂ Total Operating Expenses
   - Operating Income, Interest, Taxes ⊂ Net Income components
   If unclear whether subset or disjoint: use the LARGER single value as denominator, not the sum."""


def build_system_prompt(few_shot_examples: list[dict]) -> str:
    """Build the full system prompt with dynamic few-shot examples."""
    few_shot_block = format_few_shot(few_shot_examples)
    return f"{SYSTEM_PROMPT_BASE}\n\n## Examples\n\n{few_shot_block}"


# ── Main ask() function: 2-Step PoT Pipeline ──────────────

def ask(
    question: str,
    doc_name: str,
    top_pages: int = TOP_PAGES,
    temperature: float = 0.1,
) -> dict:
    """
    2-Step PoT pipeline:
    1. Retrieve pages (4-Way RRF)
    2. Step 1: Extract facts (thinking disabled)
    3. Step 2: Generate Python code (thinking enabled) → execute → answer
    """

    # ── 1. Encode query (all representations) ─────────
    enc = embed_model.encode(
        [question],
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=True,
    )
    query_dense = np.array(enc["dense_vecs"], dtype=np.float32)
    query_dense = query_dense / (np.linalg.norm(query_dense, axis=1, keepdims=True) + 1e-8)
    query_sparse = enc["lexical_weights"][0]
    query_colbert = enc["colbert_vecs"][0]

    # ── 2. Load document data (lazy, from DocumentStore) ──
    doc_data = doc_store.get(doc_name)
    meta = doc_data["metadata"]
    all_dense = doc_data["dense_vecs"]
    all_sparse = doc_data["sparse_vecs"]
    all_colbert = doc_data["colbert_vecs"]
    bm25 = doc_data["bm25"]
    doc_pages = doc_data["parent_pages"]

    # ── 3. Score all chunks (4 methods) ───────────────
    dense_scores = (query_dense @ all_dense.T)[0]
    bm25_scores = bm25.get_scores(question.lower().split())
    sparse_scores = np.array([sparse_sim(query_sparse, s) for s in all_sparse])
    colbert_scores = np.array([colbert_maxsim(query_colbert, c) for c in all_colbert])

    # ── 4. Aggregate to page level (best score per page) ─
    page_dense = {}
    page_bm25 = {}
    page_sparse = {}
    page_colbert = {}

    for idx in range(len(meta)):
        pg = int(meta[idx].get("page", 0))
        if pg not in page_dense or dense_scores[idx] > page_dense[pg]:
            page_dense[pg] = float(dense_scores[idx])
        if pg not in page_bm25 or bm25_scores[idx] > page_bm25[pg]:
            page_bm25[pg] = float(bm25_scores[idx])
        if pg not in page_sparse or sparse_scores[idx] > page_sparse[pg]:
            page_sparse[pg] = float(sparse_scores[idx])
        if pg not in page_colbert or colbert_scores[idx] > page_colbert[pg]:
            page_colbert[pg] = float(colbert_scores[idx])

    # ── 5. Page rankings + RRF fusion ─────────────────
    dense_ranked = sorted(page_dense, key=page_dense.get, reverse=True)
    bm25_ranked = sorted(page_bm25, key=page_bm25.get, reverse=True)
    sparse_ranked = sorted(page_sparse, key=page_sparse.get, reverse=True)
    colbert_ranked = sorted(page_colbert, key=page_colbert.get, reverse=True)

    rrf_ranked, rrf_scores = rrf_fuse([dense_ranked, bm25_ranked, sparse_ranked, colbert_ranked])

    # ── 6. Build candidate pages from parent page texts ──
    candidate_pages = []
    for pg in rrf_ranked:
        page_text = doc_pages.get(pg)
        if page_text is None:
            print(f"    WARN: page {pg} not found in parent pages for {doc_name}")
            continue
        candidate_pages.append({
            "page": pg,
            "rrf_score": round(rrf_scores[pg], 6),
            "dense_score": page_dense.get(pg, 0.0),
            "bm25_score": page_bm25.get(pg, 0.0),
            "sparse_score": page_sparse.get(pg, 0.0),
            "colbert_score": page_colbert.get(pg, 0.0),
            "text": page_text,
        })

    # ── 7. Select top pages ───────────────────────────
    selected_pages = candidate_pages[:top_pages]

    # ── 8. Build context string ───────────────────────
    context_parts = []
    for i, sp in enumerate(selected_pages):
        header = f"[Page {i+1}] page={sp['page']}"
        context_parts.append(f"{header}\n{sp['text']}")
    context_str = "\n\n---\n\n".join(context_parts)

    # ── 9. STEP 1: Fact extraction (thinking OFF) ─────
    fact_result = extract_facts(context_str, question)
    extracted_facts = fact_result["facts"]

    # ── 10. Dynamic few-shot selection ────────────────
    few_shot_examples = select_few_shot(query_dense)

    # ── 11. STEP 2: Code generation (thinking ON) ─────
    system_prompt = build_system_prompt(few_shot_examples)
    user_prompt = f"Extracted Facts:\n{extracted_facts}\n\nQuestion: {question}"

    resp = requests.post(
        LM_STUDIO_URL,
        json={
            "model": LLM_MODEL,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "temperature": temperature,
            "think": True,
        },
        timeout=1500,
    )
    resp.raise_for_status()
    resp_json = resp.json()
    msg = resp_json["choices"][0]["message"]
    llm_response = (msg.get("content") or "").strip()
    code_thinking = msg.get("reasoning_content") or None
    usage = resp_json.get("usage", {})

    # ── 12. Extract and execute code ──────────────────
    generated_code = extract_code_block(llm_response)
    retry_count = 0
    execution_error = None

    if generated_code:
        exec_result = execute_code(generated_code)
    else:
        exec_result = {"success": False, "output": "", "error": "No code block found in LLM response"}

    # ── 13. Retry on error ────────────────────────────
    if not exec_result["success"] and generated_code:
        retry_count = 1
        retry_prompt = (
            f"Your code produced an error:\n{exec_result['error']}\n\n"
            f"Original code:\n```python\n{generated_code}\n```\n\n"
            f"Fix the code. Output only the corrected ```python code block."
        )

        resp2 = requests.post(
            LM_STUDIO_URL,
            json={
                "model": LLM_MODEL,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                    {"role": "assistant", "content": llm_response},
                    {"role": "user", "content": retry_prompt},
                ],
                "temperature": temperature,
                "think": True,
            },
            timeout=1500,
        )
        resp2.raise_for_status()
        resp2_json = resp2.json()
        msg2 = resp2_json["choices"][0]["message"]
        llm_response2 = (msg2.get("content") or "").strip()
        retry_thinking = msg2.get("reasoning_content") or None
        if retry_thinking:
            code_thinking = (code_thinking or "") + "\n---RETRY---\n" + retry_thinking
        usage2 = resp2_json.get("usage", {})

        usage["prompt_tokens"] = usage.get("prompt_tokens", 0) + usage2.get("prompt_tokens", 0)
        usage["completion_tokens"] = usage.get("completion_tokens", 0) + usage2.get("completion_tokens", 0)

        code_block2 = extract_code_block(llm_response2)
        if code_block2:
            generated_code = code_block2
            exec_result = execute_code(code_block2)

    code_output = exec_result["output"] if exec_result["success"] else "CODE_ERROR"
    if not exec_result["success"]:
        execution_error = exec_result["error"]

    # ── 14. Return everything ─────────────────────────
    return {
        "question": question,
        "doc_name": doc_name,
        "llm_answer": llm_response,
        "generated_code": generated_code,
        "code_output": code_output,
        "execution_error": execution_error,
        "retry_count": retry_count,
        "model": LLM_MODEL,
        "retrieval_method": "rrf_4way_2step_pot",
        "top_pages": top_pages,
        "candidate_pages": [
            {"page": cp["page"], "rrf_score": cp["rrf_score"],
             "dense_score": cp["dense_score"], "bm25_score": cp["bm25_score"],
             "sparse_score": cp["sparse_score"], "colbert_score": cp["colbert_score"]}
            for cp in candidate_pages[:20]
        ],
        "selected_pages": [sp["page"] for sp in selected_pages],
        "context_pages": [{"page": sp["page"], "text": sp["text"]} for sp in selected_pages],
        "context_chars": len(context_str),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "extracted_facts": extracted_facts,
        "extracted_facts_raw": fact_result["raw_facts"],
        "fact_thinking": fact_result["thinking"],
        "code_thinking": code_thinking,
        "fact_extraction_tokens": {
            "prompt": fact_result["prompt_tokens"],
            "completion": fact_result["completion_tokens"],
        },
        "few_shot_questions": [ex["question"] for ex in few_shot_examples],
    }


def save_result(result: dict, q_uid: str, gt_answers: list,
                gt_pdf: str, run_file: Path,
                display_answer: str = None,
                exact_match: bool = None, lenient_match: bool = None,
                retrieval_hit: bool = None, gt_page: int = None,
                gt_page_rank: int = None, gt_page_rrf_score: float = None,
                top1_rrf_score: float = None,
                retrieved_pages: list = None) -> None:
    """Append a single result as one JSONL line."""
    record = {
        "q_uid": q_uid,
        "gt_pdf": gt_pdf,
        "gt_answers": gt_answers,
        "display_answer": display_answer,
        "exact_match": exact_match,
        "lenient_match": lenient_match,
        "retrieval_hit": retrieval_hit,
        "gt_page": gt_page,
        "gt_page_rank": gt_page_rank,
        "gt_page_rrf_score": gt_page_rrf_score,
        "top1_rrf_score": top1_rrf_score,
        "retrieved_pages": retrieved_pages,
        "timestamp": datetime.now().isoformat(),
        **result,
    }
    record.pop("llm_answer", None)
    if lenient_match:
        record.pop("context_pages", None)
        record.pop("generated_code", None)
    with jsonlines.open(run_file, mode="a") as writer:
        writer.write(record)


print("2-Step PoT Pipeline ready")
print(f"  Answer LLM:  {LLM_MODEL} @ {LM_STUDIO_URL}")
print(f"  Mode:        2-Step PoT (Fact Extraction → Code Generation → Execution)")
print(f"  Step 1:      Fact extraction (think=OFF, no reasoning)")
print(f"  Step 2:      Code generation (think=ON) with {NUM_FEW_SHOT} dynamic few-shot examples → exec()")
print(f"  Retry:       1x on execution error (think=ON)")
print(f"  Retrieval:   4-Way RRF (Dense + BM25 + Sparse + ColBERT)  |  RRF k={RRF_K}")
print(f"  Top pages:   {TOP_PAGES}  (no reranking)")
print(f"  Results:     {RESULTS_DIR}")

2-Step PoT Pipeline ready
  Answer LLM:  google/gemma-4-26b-a4b @ http://localhost:1234/v1/chat/completions
  Mode:        2-Step PoT (Fact Extraction → Code Generation → Execution)
  Step 1:      Fact extraction (think=OFF, no reasoning)
  Step 2:      Code generation (think=ON) with 3 dynamic few-shot examples → exec()
  Retry:       1x on execution error (think=ON)
  Retrieval:   4-Way RRF (Dense + BM25 + Sparse + ColBERT)  |  RRF k=60
  Top pages:   10  (no reranking)
  Results:     C:\Users\phili\PycharmProjects\Retrieval_Program_of_Thought\data\results


In [6]:
# ── DEBUG: Check LM Studio message structure ──
# Verify whether output lands in "content" or "reasoning_content"
import requests as _req

test_resp = _req.post(
    LM_STUDIO_URL,
    json={
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": "Extract numerical facts as bullet points."},
            {"role": "user", "content": "What is 2+2?"},
        ],
        "temperature": 0.1,
        "max_tokens": 500,
    },
    timeout=60,
)
test_resp.raise_for_status()
msg = test_resp.json()["choices"][0]["message"]
print("Keys in message:", list(msg.keys()))
print("content len:", len(msg.get("content") or ""))
print("reasoning_content len:", len(msg.get("reasoning_content") or ""))
if msg.get("reasoning_content"):
    print("\nreasoning_content preview:", msg["reasoning_content"][:300])
if msg.get("content"):
    print("\ncontent preview:", msg["content"][:300])

Keys in message: ['role', 'content', 'reasoning_content', 'tool_calls']
content len: 26
reasoning_content len: 368

reasoning_content preview: 
*   Input: "What is 2+2?"
    *   Task: Extract numerical facts as bullet points.

    *   The question asks for the sum of 2 and 2.
    *   The mathematical fact is that $2 + 2 = 4$.

    *   Fact 1: The first number is 2.
    *   Fact 2: The second number is 2.
    *   Fact 3: The sum is 4.

    

content preview: * The sum of 2 and 2 is 4.


In [7]:
# ── Run ALL benchmark questions (2-Step PoT mode) ──────
from datetime import datetime

SHOW_PAGES = 20

run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
model_tag = LLM_MODEL.split("/")[-1]
run_file = RESULTS_DIR / f"run_fin_2step_pot_{model_tag}_top{TOP_PAGES}_{run_ts}.jsonl"
run_json_file = RESULTS_DIR / f"run_fin_2step_pot_{model_tag}_top{TOP_PAGES}_{run_ts}.json"

all_results = []
total_exact = 0
total_lenient = 0
total_retrieval_hit = 0
total_retrieval_eval = 0
total_retries = 0
total_code_errors = 0
skipped = 0

for i, row in df.iterrows():
    # Skip docs without parent pages
    if not doc_store.has_parent_pages(row["doc_name"]):
        skipped += 1
        print(f"── Q{i+1}/{len(df)}  SKIP doc={row['doc_name']} (no parent pages) ──")
        continue

    result = ask(
        question=row["question"],
        doc_name=row["doc_name"],
    )

    # ── Answer from code execution ────────────────────
    display_answer = result["code_output"]
    # Take last line if multiple lines printed
    if "\n" in display_answer:
        display_answer = display_answer.strip().split("\n")[-1].strip()

    # ── Track retries and errors ──────────────────────
    total_retries += result["retry_count"]
    if result["execution_error"]:
        total_code_errors += 1

    # ── Build GT answers list (skip NaN) ───────────────
    gt_answers = [str(v) for v in [row["answer_1"], row["answer_2"]] if pd.notna(v)]

    # ── Answer matching (exact + lenient) ───────────────
    match = match_answer(display_answer, gt_answers)
    total_exact += match["exact_match"]
    total_lenient += match["lenient_match"]
    # ── Retrieval hit: is GT page among selected pages? ──
    gt_page_match = re.search(r"page_(\d+)", str(row["gt_pdf"]))
    gt_page_num = int(gt_page_match.group(1)) if gt_page_match else None
    selected_pages = result["selected_pages"]
    retrieval_hit = gt_page_num in selected_pages if gt_page_num is not None else None
    if retrieval_hit is not None:
        total_retrieval_hit += retrieval_hit
        total_retrieval_eval += 1

    # GT page rank in RRF results
    gt_page_rank = None
    gt_page_rrf_score = None
    for cp_idx, cp in enumerate(result["candidate_pages"]):
        if cp["page"] == gt_page_num:
            gt_page_rank = cp_idx + 1
            gt_page_rrf_score = cp["rrf_score"]
            break
    top1_rrf_score = result["candidate_pages"][0]["rrf_score"] if result["candidate_pages"] else None

    # ── Print diagnostics ───────────────────────────────
    n = i + 1 - skipped
    hit_icon = "✓" if retrieval_hit else ("✗" if retrieval_hit is False else "?")
    exact_icon = "✓" if match["exact_match"] else "✗"
    lenient_icon = "✓" if match["lenient_match"] else "✗"
    running_exact_pct = total_exact / n * 100
    running_lenient_pct = total_lenient / n * 100

    print(f"── Q{n}/{len(df) - skipped}  doc={row['doc_name']} ──")
    print(f"Frage:          {row['question']}")
    print(f"Gold-Antwort:   {gt_answers}")
    print(f"Code-Antwort:   {display_answer}")
    rank_str = f"rank={gt_page_rank}" if gt_page_rank else "rank=miss"
    print(f"Exact: {exact_icon}  Lenient: {lenient_icon}  Retrieval: {hit_icon} {rank_str}  (GT page={gt_page_num}, selected={selected_pages})")
    print(f"Running:        exact={running_exact_pct:.1f}%  lenient={running_lenient_pct:.1f}%")

    # ── Print code execution info ─────────────────────
    code_icon = "✓" if not result["execution_error"] else "✗"
    retry_info = f"  retry={result['retry_count']}" if result["retry_count"] > 0 else ""
    err_info = f"  err={result['execution_error']}" if result["execution_error"] else ""
    print(f"Code exec:      {code_icon}{retry_info}{err_info}")
    if result["generated_code"]:
        code_preview = result["generated_code"].replace("\n", " | ")[:200]
        print(f"Code:           {code_preview}")

    # ── Print thinking info ───────────────────────────
    fact_think_len = len(result["fact_thinking"]) if result["fact_thinking"] else 0
    code_think_len = len(result["code_thinking"]) if result["code_thinking"] else 0
    print(f"Thinking:       fact={fact_think_len} chars  code={code_think_len} chars")

    # ── Print few-shot + facts summary ────────────────
    print(f"Few-shot:       {[q[:60] + '...' for q in result['few_shot_questions']]}")
    fact_toks = result["fact_extraction_tokens"]
    print(f"Facts tokens:   prompt={fact_toks['prompt']}  completion={fact_toks['completion']}")
    facts_preview = result["extracted_facts"][:200].replace("\n", " | ")
    print(f"Facts preview:  {facts_preview}")

    # ── Print candidate pages table (top 20) ───────────
    pages_to_show = result["candidate_pages"][:SHOW_PAGES]
    print(f"\n  PAGE SCORES (top {len(pages_to_show)} / {len(result['candidate_pages'])})")
    print(f"  {'─'*84}")
    print(f"  {'Page':>6}  {'RRF':>8}  {'Dense':>8}  {'BM25':>8}  {'Sparse':>8}  {'ColBERT':>8}")
    print(f"  {'─'*84}")
    for cp in pages_to_show:
        sel = "→" if cp["page"] in selected_pages else " "
        gt_mark = " << GT" if cp["page"] == gt_page_num else ""
        print(f"  {sel} {cp['page']:>4}  {cp['rrf_score']:>8.4f}  {cp['dense_score']:>8.4f}  {cp['bm25_score']:>8.4f}  {cp['sparse_score']:>8.4f}  {cp['colbert_score']:>8.2f}{gt_mark}")
    print()

    # ── Save result ───────────────────────────────────
    save_result(
        result=result,
        q_uid=row.get("q_uid", f"q_{i}"),
        gt_answers=gt_answers,
        gt_pdf=str(row.get("gt_pdf", "")),
        run_file=run_file,
        display_answer=display_answer,
        exact_match=match["exact_match"],
        lenient_match=match["lenient_match"],
        retrieval_hit=retrieval_hit,
        gt_page=gt_page_num,
        gt_page_rank=gt_page_rank,
        gt_page_rrf_score=gt_page_rrf_score,
        top1_rrf_score=top1_rrf_score,
        retrieved_pages=selected_pages,
    )

    all_results.append(result)

# ── Final summary ───────────────────────────────────
print(f"\n{'═'*60}")
evaluated = len(df) - skipped
print(f"FINAL RESULTS ({evaluated} questions, {skipped} skipped)  |  TOP_PAGES={TOP_PAGES}  |  2-Step PoT")
print(f"{'═'*60}")
print(f"  Exact match:    {total_exact}/{evaluated} ({total_exact/evaluated*100:.1f}%)")
print(f"  Lenient match:  {total_lenient}/{evaluated} ({total_lenient/evaluated*100:.1f}%)")
if total_retrieval_eval > 0:
    print(f"  Retrieval hit:  {total_retrieval_hit}/{total_retrieval_eval} ({total_retrieval_hit/total_retrieval_eval*100:.1f}%)")
print(f"  Code errors:    {total_code_errors}/{evaluated}")
print(f"  Retries:        {total_retries}/{evaluated}")
print(f"  JSONL: {run_file}")

# ── Save as JSON too ────────────────────────────
import json as _json
with open(run_json_file, "w", encoding="utf-8") as f:
    _json.dump(all_results, f, ensure_ascii=True, indent=2)
print(f"  JSON:  {run_json_file}")

── Q1/895  doc=AAL_2017 ──
Frage:          did american have access to more planes than american eagle at 12/31/17?
Gold-Antwort:   ['yes', 'yes']
Code-Antwort:   yes
Exact: ✓  Lenient: ✓  Retrieval: ✗ rank=11  (GT page=33, selected=[45, 5, 169, 34, 120, 196, 39, 6, 7, 166])
Running:        exact=100.0%  lenient=100.0%
Code exec:      ✓
Code:           mainline_aircraft = 948 | regional_aircraft = 597 |  | if mainline_aircraft > regional_aircraft: |     print('yes') | else: |     print('no')
Thinking:       fact=1198 chars  code=509 chars
Few-shot:       ['what was the difference in millions of international subscri...', 'what is the percentage change in the balance of non-controll...', 'what was the percentage cumulative total shareholder return ...']
Facts tokens:   prompt=9400  completion=417
Facts preview:  FACTS: | - Mainline aircraft operated as of December 31, 2017 : 948 aircraft | - Regional aircraft operated as of December 31, 2017 : 597 aircraft

  PAGE SCORES (top 20 / 20)
 

HTTPError: 400 Client Error: Bad Request for url: http://localhost:1234/v1/chat/completions